
# TF‑IDF + XGBoost: Cleaning Comparison (Gensim‑style vs Regex+NLTK‑style)

This notebook does the following:
1. **Loads** the BBC Text dataset.
2. **Cleans** text using two approaches:
   - *Gensim‑style* (lowercase → strip punctuation/numbers → stopwords → drop short tokens)
   - *Regex+NLTK‑style* (lowercase → remove HTML tags → punctuation/numbers → stopwords → light stemming)
3. **Vectorizes** with `TfidfVectorizer`.
4. **Trains** two XGBoost models (one per cleaning pipeline).
5. **Evaluates** each (accuracy, precision, recall, F1, confusion matrix).
6. **Compares** results side‑by‑side.


In [ ]:

import os, re, string
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# XGBoost (scikit-learn API)
try:
    import xgboost as xgb
except Exception as e:
    raise ImportError("XGBoost is required for this notebook. Please install `xgboost`.") from e

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Load BBC dataset

In [ ]:

# The dataset is expected to be present at this path.
data_path = Path('/mnt/data/bbc-text.csv')
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path}. Place 'bbc-text.csv' there.")

df = pd.read_csv(data_path)
df.head()


## Cleaning Pipelines

In [ ]:

# A compact English stopword set (self-contained for portability)
STOPWORDS = {
    "a","an","the","and","or","but","if","while","with","without","of","at","by","for","from","into","onto","to","in","on","off","out","up","down",
    "as","is","are","was","were","be","been","being","am","do","does","did","doing","have","has","had","having",
    "that","this","these","those","it","its","itself","they","them","their","theirs","themselves","he","him","his","himself",
    "she","her","hers","herself","you","your","yours","yourself","yourselves","we","us","our","ours","ourselves",
    "i","me","my","mine","myself","not","no","nor","so","than","too","very","can","could","should","would","may","might","must","will","shall",
    "about","above","after","again","against","all","any","both","each","few","more","most","other","some","such","only","own","same","then","once",
    "because","until","between","over","under","why","how","what","who","whom","which","when","where"
}

PUNCT_TABLE = str.maketrans("", "", string.punctuation)
TAG_RE = re.compile(r"<.*?>")

def simple_stem(word: str) -> str:
    """A lightweight, rule-based stemmer (portable fallback)."""
    for suf in ("ing", "edly", "ed", "ly", "ies", "s"):
        if word.endswith(suf) and len(word) - len(suf) >= 3:
            if suf == "ies":
                return word[:-3] + "y"
            return word[: -len(suf)]
    return word

def clean_gensim_style(text: str) -> str:
    """Lowercase → strip punctuation → strip numbers → normalize spaces → remove stopwords → drop short tokens (<3)."""
    text = text.lower()
    text = text.translate(PUNCT_TABLE)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if (t not in STOPWORDS and len(t) >= 3)]
    return " ".join(tokens)

def clean_regex_nltk_style(text: str) -> str:
    """Lowercase → remove HTML tags → strip punctuation/numbers → normalize spaces → stopwords → lightweight stemming."""
    text = text.lower()
    text = TAG_RE.sub(" ", text)
    text = text.translate(PUNCT_TABLE)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if t not in STOPWORDS]
    tokens = [simple_stem(t) for t in tokens]
    return " ".join(tokens)


In [ ]:

df_gensim = df.copy()
df_regex  = df.copy()

df_gensim['clean'] = df_gensim['text'].apply(clean_gensim_style)
df_regex['clean']  = df_regex['text'].apply(clean_regex_nltk_style)

df_gensim[['category','clean']].head(5)


## Vectorize, Train, and Evaluate

In [ ]:

def vectorize(series, max_features=20000, ngram_range=(1,2)):
    vec = TfidfVectorizer(stop_words='english', max_features=max_features, ngram_range=ngram_range)
    X = vec.fit_transform(series)
    return vec, X

def train_xgb(X_train, y_train, num_class):
    clf = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softprob',
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=RANDOM_STATE
    )
    clf.fit(X_train, y_train)
    return clf

def evaluate_and_plot(model, X_test, y_test, title="Model"):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)

    print(f"\n=== {title} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {pr:.4f}  Recall: {rc:.4f}  F1: {f1:.4f}\n")
    print(classification_report(y_test, y_pred, zero_division=0))

    # Confusion Matrix
    fig = plt.figure(figsize=(6, 5))
    disp = ConfusionMatrixDisplay.from_predictions(y_test, y_pred, xticks_rotation='vertical')
    plt.title(f"Confusion Matrix — {title}")
    plt.tight_layout()
    plt.show()

    return {"accuracy": acc, "precision_w": pr, "recall_w": rc, "f1_w": f1}


In [ ]:

# Prepare labels
y = df['category']

# --- Gensim-style ---
vec_gensim, X_gensim = vectorize(df_gensim['clean'])
Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    X_gensim, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# --- Regex+NLTK-style ---
vec_regex, X_regex = vectorize(df_regex['clean'])
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_regex, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

Xg_train.shape, Xr_train.shape


In [ ]:

# Train models
model_gensim = train_xgb(Xg_train, yg_train, num_class=y.nunique())
model_regex  = train_xgb(Xr_train, yr_train, num_class=y.nunique())
"Training complete."


In [ ]:

# Evaluate
metrics_gensim = evaluate_and_plot(model_gensim, Xg_test, yg_test, title="XGBoost (Gensim-style cleaned)")
metrics_regex  = evaluate_and_plot(model_regex,  Xr_test, yr_test, title="XGBoost (Regex+NLTK-style cleaned)")


## Side-by-Side Metrics

In [ ]:

cmp = pd.DataFrame([
    {"pipeline": "Gensim-style", **metrics_gensim},
    {"pipeline": "Regex+NLTK-style", **metrics_regex},
]).set_index('pipeline')
cmp
